In [10]:
import os
while not os.path.isdir("DataTypes"):
    os.chdir("..")

load("DataTypes/Grafo2Terminales.sage")
load("Scripts/NumeroDe2Bosques.sage")
load("Scripts/GrafoBalloon.sage")
load("Scripts/CantidadesEstructurales.sage")

In [11]:
def MaximoT2EnT(n, m):
    """
        Recorre todos los grafos conexos de n vértices y m aristas
        (C_{n,m}, vía nauty) y todas las ubicaciones de 2 terminales
        s, t; construye cada Grafo2Terminales y calcula su t2.
        Retorna (max t2, cantidad de grafos, cantidad de pares (G, {s,t}),
        un Grafo2Terminales óptimo).
    """
    t2_max, optimo = -1, None
    cant_grafos, cant_pares = 0, 0
    for G in GenerarGrafosConexos(n, m):
        cant_grafos += 1
        V = G.vertices()
        for i in range(n):
            for j in range(i + 1, n):
                TC = Grafo2Terminales(G, V[i], V[j])
                valor = NumeroDe2Bosques(TC)
                cant_pares += 1
                if valor > t2_max:
                    t2_max, optimo = valor, TC
    return t2_max, cant_grafos, cant_pares, optimo

In [12]:
# n = 11 ya requiere ~7 millones de grafos (horas) y n = 12 es inviable por fuerza bruta
resultados = {}
for n in range(7, 7 + 1):
    for m in [binomial(n - 2, 2) + 1, binomial(n - 2, 2) + 2]:
        t2_G = NumeroDe2Bosques(ConstruirGrafo2TerminalesBalloon(n, m))
        t2_max, cant_grafos, cant_pares, optimo = MaximoT2EnT(n, m)
        resultados[(n, m)] = optimo
        print(f"n={n}, m={m}: |C|={cant_grafos}, pares={cant_pares}, t2(G)={t2_G}, "
              f"max t2(H)={t2_max}, t2(G)/max t2(H)={t2_G / t2_max} ~ {float(t2_max / t2_G):.4f}")

n=7, m=11: |C|=138, pares=2898, t2(G)=200, max t2(H)=225, t2(G)/max t2(H)=8/9 ~ 1.1250
n=7, m=12: |C|=126, pares=2646, t2(G)=300, max t2(H)=384, t2(G)/max t2(H)=25/32 ~ 1.2800


In [13]:
def MaximizantesT2EnT(n, m):
    """
        Igual que MaximoT2EnT pero retorna (max t2, lista de TODOS los
        Grafo2Terminales que alcanzan el máximo). Los pares (G, {s,t})
        equivalentes por isomorfismo (que fijan s,t) se listan una sola vez.
    """
    t2_max, maximizantes, vistos = -1, [], set()
    for G in GenerarGrafosConexos(n, m):
        V = G.vertices()
        for i in range(n):
            for j in range(i + 1, n):
                TC = Grafo2Terminales(G, V[i], V[j])
                valor = NumeroDe2Bosques(TC)
                if valor < t2_max:
                    continue
                if valor > t2_max:
                    t2_max, maximizantes, vistos = valor, [], set()
                clave = G.canonical_label(partition=[[V[i], V[j]], [v for v in V if v not in (V[i], V[j])]]).graph6_string()
                if clave not in vistos:
                    vistos.add(clave)
                    maximizantes.append(TC)
    return t2_max, maximizantes


import json

ARCHIVO_MAXIMIZANTES = "Datos/MaximizantesT2.json"
INICIO, FIN = 7, 10  # rango de n (ambos inclusive)

def CargarMaximizantes(archivo=ARCHIVO_MAXIMIZANTES):
    """
        Lee el archivo y retorna el diccionario {n: {m: {"t2_max": int,
        "maximizantes": [{"s", "t", "graph6"}, ...]}}} con claves enteras.
        Para reconstruir un óptimo: Grafo2Terminales(Graph(d["graph6"]), d["s"], d["t"]).
    """
    if not os.path.exists(archivo):
        return {}
    with open(archivo) as f:
        crudo = json.load(f)
    return {int(n): {int(m): d for m, d in por_m.items()} for n, por_m in crudo.items()}

def GuardarMaximizantes(datos, archivo=ARCHIVO_MAXIMIZANTES):
    os.makedirs(os.path.dirname(archivo), exist_ok=True)
    with open(archivo, "w") as f:
        json.dump(datos, f, indent=1)

maximizantes_por_nm = CargarMaximizantes()
for n in range(INICIO, FIN + 1):
    for m in [binomial(n - 2, 2) + 1, binomial(n - 2, 2) + 2]:
        t2_max, maximizantes = MaximizantesT2EnT(n, m)
        print(f"n={n}, m={m}: max t2 = {t2_max}, #maximizantes (salvo isomorfismo) = {len(maximizantes)}")
        for k, TC in enumerate(maximizantes, 1):
            print(f"  {k}) s={TC.s}, t={TC.t}, aristas={sorted(TC.grafo.edges(labels=False))}")
        maximizantes_por_nm.setdefault(int(n), {})[int(m)] = {
            "t2_max": int(t2_max),
            "maximizantes": [{"s": int(TC.s), "t": int(TC.t), "graph6": TC.grafo.graph6_string()}
                             for TC in maximizantes],
        }
        GuardarMaximizantes(maximizantes_por_nm)  # se guarda tras cada (n, m) por si se interrumpe


n=7, m=11: max t2 = 225, #maximizantes (salvo isomorfismo) = 7
  1) s=1, t=3, aristas=[(0, 3), (0, 4), (0, 5), (0, 6), (1, 4), (1, 5), (1, 6), (2, 4), (2, 5), (2, 6), (4, 6)]
  2) s=2, t=3, aristas=[(0, 3), (0, 4), (0, 5), (0, 6), (1, 4), (1, 5), (1, 6), (2, 5), (2, 6), (3, 4), (4, 6)]
  3) s=2, t=3, aristas=[(0, 3), (0, 4), (0, 5), (0, 6), (1, 4), (1, 5), (1, 6), (2, 5), (3, 4), (3, 6), (4, 6)]
  4) s=2, t=3, aristas=[(0, 3), (0, 4), (0, 5), (0, 6), (1, 4), (1, 5), (1, 6), (2, 5), (2, 6), (3, 4), (5, 6)]
  5) s=2, t=3, aristas=[(0, 3), (0, 4), (0, 5), (0, 6), (1, 4), (1, 5), (1, 6), (2, 5), (3, 4), (3, 6), (5, 6)]
  6) s=1, t=2, aristas=[(0, 3), (0, 4), (0, 5), (0, 6), (1, 4), (2, 5), (2, 6), (3, 4), (3, 5), (3, 6), (4, 6)]
  7) s=3, t=4, aristas=[(0, 2), (0, 4), (0, 5), (0, 6), (1, 3), (1, 5), (1, 6), (2, 4), (2, 5), (2, 6), (5, 6)]
n=7, m=12: max t2 = 384, #maximizantes (salvo isomorfismo) = 3
  1) s=2, t=3, aristas=[(0, 3), (0, 4), (0, 5), (0, 6), (1, 3), (1, 4), (1, 5), (1, 6), (2